# Build a unified full-trajectory math/proof solver dataset

This notebook downloads and normalizes several math/proof datasets into one JSONL file for **full trajectory / full solution generation**.

## Target output schema

Each output example has this shape:

```json
{
  "problem": "...",
  "solution_steps": ["step 1", "step 2", "step 3"],
  "final_answer": "...",
  "example_index": 137,
  "source": "proofnet"
}
```

## Datasets handled here

- **DEMI-MathAnalysis**: expected as a local JSONL file you already exported.
- **ProofNet**: Hugging Face dataset `hoskinson-center/proofnet`.
- **NaturalProofs generation data**: Hugging Face dataset `wellecks/naturalproofs-gen`.
- **MATH**: Hugging Face dataset `EleutherAI/hendrycks_math`.


## Some Notes

- Some source datasets are **proof-style** rather than answer-style. In those cases, `final_answer` may be empty.
- Solution steps are extracted heuristically from the original proof/solution text.


In [38]:
# Install older datasets to support dataset python scripts (removed in v3.0+)
!pip -q install "datasets<3.0" pandas tqdm

In [39]:

from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

from datasets import load_dataset
from tqdm.auto import tqdm
import pandas as pd

# Compatibility patch: datasets<3.0 can hit a dill/pickle legacy-cache bug on Python 3.14+.
# The legacy cache probe is only for reusing old cache locations, so disabling it is safe here.
def patch_datasets_legacy_cache_for_python314() -> None:
    import sys
    if sys.version_info < (3, 14):
        return
    try:
        from datasets.builder import DatasetBuilder
    except Exception:
        return

    def _skip_legacy_cache2(self, dataset_module):
        return None

    DatasetBuilder._check_legacy_cache2 = _skip_legacy_cache2


patch_datasets_legacy_cache_for_python314()


In [40]:
# Configure folders and file paths

OUTPUT_DIR = Path("normalized_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = OUTPUT_DIR / "solver_full_trajectory_dataset.jsonl"

USE_DEMI = True
USE_PROOFNET = True
USE_NATURALPROOFS = True
USE_MATH = True

# we want a mix of the sources, so we broke up the sources like so.
TARGET_SOURCE_COUNTS = {
    "math": 1500,
    "naturalproofs": 225,
    "proofnet": 225,
    "demi_mathanalysis": 50,
}

TARGET_TOTAL_PROBLEMS = sum(TARGET_SOURCE_COUNTS.values())

# MATH split choices:
MATH_SPLITS = ["train", "test"]

print(f"Normalized output file will be written to: {OUTPUT_JSONL.resolve()}")


Normalized output file will be written to: /Users/timli/MathSolverLLMs/normalized_outputs/solver_full_trajectory_dataset.jsonl


In [41]:
# Utility functions for processing and normalizing the datasets

from typing import Any, Dict, List, Optional
import re

def first_present(d: Dict[str, Any], keys: List[str], default=None):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return default

def clean_text(x: Any) -> str:
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\r\n", "\n").replace("\r", "\n")
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\n{3,}", "\n\n", x)
    return x.strip()

def split_solution_into_steps(text: str) -> List[str]:
    """
    Heuristic step splitter:
    1. preserve numbered / bulleted lines
    2. otherwise split on double newlines
    3. fallback to sentence-ish chunks if needed
    """
    text = clean_text(text)
    if not text:
        return []

    # Normalize LaTeX line breaks a bit
    text = text.replace("\\\\\n", "\n").replace("\\\\", "\n")

    # Try line-based numbered/bulleted steps
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    numbered_like = []
    for ln in lines:
        if re.match(r"^(\(?\d+[\).\:]|[-*\u2022]|Step\s+\d+[:.\-])", ln, flags=re.I):
            numbered_like.append(re.sub(r"\s+", " ", ln).strip())

    if len(numbered_like) >= 2:
        return numbered_like

    # Split on paragraph breaks
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if len(paras) >= 2:
        return paras

    # Sentence-ish fallback
    sents = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\\(])", text)
    sents = [s.strip() for s in sents if s.strip()]
    if len(sents) >= 2:
        return sents

    return [text]

def extract_boxed_answer(text: str) -> str:
    text = clean_text(text)
    if not text:
        return ""

    # \boxed{...}
    m = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if m:
        return m[-1].strip()

    # final answer is ...
    patterns = [
        r"(?i)final answer\s*(?:is|:)\s*(.+)$",
        r"(?i)therefore[, ]+the answer\s*(?:is|:)\s*(.+)$",
        r"(?i)thus[, ]+the answer\s*(?:is|:)\s*(.+)$",
        r"(?i)answer\s*[:]\s*(.+)$",
    ]
    for pat in patterns:
        mm = re.search(pat, text, flags=re.M)
        if mm:
            return mm.group(1).strip().rstrip(".")

    return ""

def maybe_add(item: Optional[Dict[str, Any]], acc: List[Dict[str, Any]]):
    if item is None:
        return
    if not item.get("problem"):
        return
    if not item.get("solution_steps"):
        return
    acc.append(item)

def preview_dataset_dict(ds, name: str, n: int = 3):
    print(f"\n===== {name} =====")
    print(ds)
    if hasattr(ds, "keys"):
        for split in ds.keys():
            row = ds[split][0]
            print(f"Split={split}, columns={list(row.keys())}")
            break

## Source-specific normalizers

In [42]:
def normalize_demi_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    """
    Tries to normalize a local DEMI JSONL export.
    Adjust the field mappings below if your local file uses different names.
    """
    problem = first_present(rec, ["problem", "question", "prompt", "input"])
    solution = first_present(
        rec,
        [
            "ground_truth_solution",
            "solution",
            "proof",
            "response",
            "output",
            "target",
            "completion",
            "answer",
        ],
    )
    final_answer = first_present(
        rec,
        ["final_answer", "ground_truth_answer", "answer_only", "short_answer"],
        default="",
    )

    problem = clean_text(problem)
    solution = clean_text(solution)
    final_answer = clean_text(final_answer) or extract_boxed_answer(solution)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(solution),
        "final_answer": final_answer,
        "example_index": example_index,
        "source": "demi_mathanalysis",
    }

def normalize_proofnet_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    problem = first_present(
        rec,
        [
            "nl_statement",
            "informal_statement",
            "natural_language_statement",
            "statement",
            "theorem",
        ],
    )
    proof = first_present(
        rec,
        [
            "nl_proof",
            "informal_proof",
            "natural_language_proof",
            "proof",
        ],
    )
    problem = clean_text(problem)
    proof = clean_text(proof)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(proof),
        "final_answer": "",
        "example_index": example_index,
        "source": "proofnet",
    }

def normalize_naturalproofs_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    """
    naturalproofs-gen has changed across versions / views, so this function is intentionally flexible.
    """
    problem = first_present(
        rec,
        [
            "statement",
            "theorem",
            "question",
            "prompt",
            "target_statement",
            "nl_statement",
            "text",
        ],
    )
    proof = first_present(
        rec,
        [
            "proof",
            "target",
            "response",
            "completion",
            "generated_proof",
            "nl_proof",
        ],
    )

    # Some variants package data in nested fields or instruction formats.
    if not problem:
        src = first_present(rec, ["source", "input", "context"], default="")
        if isinstance(src, dict):
            problem = first_present(src, ["statement", "theorem", "question", "prompt"])
    if not proof:
        tgt = first_present(rec, ["output", "answer"], default="")
        if isinstance(tgt, dict):
            proof = first_present(tgt, ["proof", "response", "target"])

    problem = clean_text(problem)
    proof = clean_text(proof)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(proof),
        "final_answer": "",
        "example_index": example_index,
        "source": "naturalproofs",
    }

def normalize_math_record(rec: Dict[str, Any], example_index: int) -> Optional[Dict[str, Any]]:
    problem = clean_text(first_present(rec, ["problem", "question"]))
    solution = clean_text(first_present(rec, ["solution", "answer"]))
    final_answer = clean_text(first_present(rec, ["final_answer", "answer_only"], default=""))
    final_answer = final_answer or extract_boxed_answer(solution)

    return {
        "problem": problem,
        "solution_steps": split_solution_into_steps(solution),
        "final_answer": final_answer,
        "example_index": example_index,
        "source": "math",
        "level": first_present(rec, ["level"], default=""),
        "category": first_present(rec, ["type", "subject", "category"], default=""),
    }


## Load each dataset

In [43]:

all_records = []
source_counts = {}
global_index = 0


In [44]:

# DEMI-MathAnalysis is a smaller dataset but has very high-quality step-by-step solutions,
# so we include it in its entirety (up to the target count) without filtering by difficulty or category.
# Adjust the normalization function if your local export has different field names or structure.
import subprocess
import json
import ast
from pathlib import Path

if USE_DEMI:
    demi_records = []
    print("Cloning DEMI-MathAnalysis repository...")
    # Clone the repo into /tmp
    subprocess.run(["git", "clone", "https://github.com/ziye2chen/DEMI-MathAnalysis.git", "/tmp/DEMI-MathAnalysis"], capture_output=True)

    repo_path = Path("/tmp/DEMI-MathAnalysis")
    # Search for any json or jsonl files
    data_files = list(repo_path.rglob("*.json")) + list(repo_path.rglob("*.jsonl"))
    print(f"Found data files: {[f.name for f in data_files]}")

    for data_file in data_files:
        try:
            with open(data_file, "r", encoding="utf-8") as f:
                content = f.read().strip()
                if content.startswith("["):
                    # Parse as a single JSON array
                    try:
                        records = json.loads(content)
                    except json.JSONDecodeError:
                        records = ast.literal_eval(content)
                    demi_records.extend(records)
                else:
                    # Try parsing the entire content as a single dictionary first
                    try:
                        try:
                            record = json.loads(content)
                        except json.JSONDecodeError:
                            record = ast.literal_eval(content)
                        if isinstance(record, dict):
                            demi_records.append(record)
                        else:
                            raise ValueError("Not a single dictionary")
                    except Exception:
                        # Parse as JSONL
                        for line in content.split('\n'):
                            if line.strip():
                                try:
                                    demi_records.append(json.loads(line))
                                except json.JSONDecodeError:
                                    demi_records.append(ast.literal_eval(line))
        except Exception as e:
            print(f"Could not parse {data_file.name}: {e}")

    demi_target = TARGET_SOURCE_COUNTS["demi_mathanalysis"]
    added_demi = 0
    seen_demi_problems = set()
    for rec in tqdm(demi_records, desc="Normalizing DEMI"):
        if added_demi >= demi_target:
            break
        item = normalize_demi_record(rec, global_index)
        if item and item.get("problem") and item.get("solution_steps"):
            prob_key = item["problem"]
            if prob_key not in seen_demi_problems:
                seen_demi_problems.add(prob_key)
                all_records.append(item)
                global_index += 1
                added_demi += 1

    source_counts["demi_mathanalysis"] = len([r for r in all_records if r["source"] == "demi_mathanalysis"])
    print("Loaded DEMI examples:", source_counts["demi_mathanalysis"])
else:
    source_counts["demi_mathanalysis"] = 0
    print("Skipping DEMI.")


Cloning DEMI-MathAnalysis repository...
Found data files: ['problem_13.json', 'problem_44.json', 'problem_52.json', 'problem_29.json', 'problem_68.json', 'problem_87.json', 'problem_48.json', 'problem_72.json', 'problem_25.json', 'problem_33.json', 'problem_64.json', 'problem_65.json', 'problem_32.json', 'problem_24.json', 'problem_73.json', 'problem_49.json', 'problem_69.json', 'problem_86.json', 'problem_28.json', 'problem_1.json', 'problem_53.json', 'problem_45.json', 'problem_12.json', 'problem_19.json', 'problem_58.json', 'problem_74.json', 'problem_23.json', 'problem_35.json', 'problem_62.json', 'problem_15.json', 'problem_42.json', 'problem_54.json', 'problem_6.json', 'problem_78.json', 'problem_39.json', 'problem_81.json', 'problem_80.json', 'problem_38.json', 'problem_79.json', 'problem_7.json', 'problem_55.json', 'problem_43.json', 'problem_14.json', 'problem_63.json', 'problem_34.json', 'problem_22.json', 'problem_75.json', 'problem_59.json', 'problem_18.json', 'problem_60.j

Normalizing DEMI:   0%|          | 0/176 [00:00<?, ?it/s]

Loaded DEMI examples: 50


In [45]:
# ProofNet is a newer dataset of informal mathematical statements and proofs. It has a lot of examples, so we filter to the target count without additional difficulty or category filtering.

if USE_PROOFNET:
    proofnet = load_dataset("hoskinson-center/proofnet", trust_remote_code=True)
    preview_dataset_dict(proofnet, "ProofNet")

    proofnet_target = TARGET_SOURCE_COUNTS["proofnet"]
    added_proofnet = 0
    seen_proofnet_problems = set()
    for split in proofnet.keys():
        if added_proofnet >= proofnet_target:
            break
        rows = proofnet[split]
        for rec in tqdm(rows, desc=f"Normalizing ProofNet/{split}"):
            if added_proofnet >= proofnet_target:
                break
            item = normalize_proofnet_record(rec, global_index)
            if item and item.get("problem") and item.get("solution_steps"):
                prob_key = item["problem"]
                if prob_key not in seen_proofnet_problems:
                    seen_proofnet_problems.add(prob_key)
                    all_records.append(item)
                    global_index += 1
                    added_proofnet += 1

    source_counts["proofnet"] = len([r for r in all_records if r["source"] == "proofnet"])
    print("Loaded ProofNet examples:", source_counts["proofnet"])
else:
    source_counts["proofnet"] = 0



===== ProofNet =====
DatasetDict({
    test: Dataset({
        features: ['id', 'nl_statement', 'nl_proof', 'formal_statement', 'src_header'],
        num_rows: 186
    })
    validation: Dataset({
        features: ['id', 'nl_statement', 'nl_proof', 'formal_statement', 'src_header'],
        num_rows: 185
    })
})
Split=test, columns=['id', 'nl_statement', 'nl_proof', 'formal_statement', 'src_header']


Normalizing ProofNet/test:   0%|          | 0/186 [00:00<?, ?it/s]

Normalizing ProofNet/validation:   0%|          | 0/185 [00:00<?, ?it/s]

Loaded ProofNet examples: 225


In [ ]:
# NaturalProofs is a large dataset of natural language mathematical problems and proofs.
# It has multiple versions and views, so the normalization function is designed to be flexible to handle different field names and structures.
# We filter to the target count without additional difficulty or category filtering, but we enforce uniqueness on the problem statements to avoid duplicates across versions.

naturalproofs_limit = TARGET_SOURCE_COUNTS["naturalproofs"]

if USE_NATURALPROOFS and naturalproofs_limit > 0:
    # Added verification_mode="no_checks" to bypass metadata mismatch errors without warnings
    naturalproofs = load_dataset("wellecks/naturalproofs-gen", trust_remote_code=True, verification_mode="no_checks")
    preview_dataset_dict(naturalproofs, "NaturalProofs")

    added_np = 0
    seen_np_problems = set()
    for split in naturalproofs.keys():
        rows = naturalproofs[split]
        for rec in tqdm(rows, desc=f"Normalizing NaturalProofs/{split}"):
            if added_np >= naturalproofs_limit:
                break
            item = normalize_naturalproofs_record(rec, global_index)

            # Check if valid item before adding, and enforce uniqueness on the fly
            if item and item.get("problem") and item.get("solution_steps"):
                prob_key = item["problem"]
                if prob_key not in seen_np_problems:
                    seen_np_problems.add(prob_key)
                    all_records.append(item)
                    global_index += 1
                    added_np += 1

        if added_np >= naturalproofs_limit:
            break

    source_counts["naturalproofs"] = len([r for r in all_records if r["source"] == "naturalproofs"])
    print("Loaded NaturalProofs examples:", source_counts["naturalproofs"])
else:
    source_counts["naturalproofs"] = 0
    print(f"Skipping NaturalProofs. (limit: {naturalproofs_limit})")



===== NaturalProofs =====
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'text', 'target', 'ctxs'],
        num_rows: 12424
    })
    validation: Dataset({
        features: ['id', 'title', 'text', 'target', 'ctxs'],
        num_rows: 1139
    })
    test: Dataset({
        features: ['id', 'title', 'text', 'target', 'ctxs'],
        num_rows: 1135
    })
})
Split=train, columns=['id', 'title', 'text', 'target', 'ctxs']


Normalizing NaturalProofs/train:   0%|          | 0/12424 [00:00<?, ?it/s]

Loaded NaturalProofs examples: 225


In [ ]:
# Finally, we have the MATH dataset, which has a lot of examples across various categories and difficulties.
# We filter to the target count without additional filtering, but we enforce uniqueness on the problem statements to avoid duplicates across categories and difficulties.

remaining_for_math = TARGET_SOURCE_COUNTS["math"]

if USE_MATH and remaining_for_math > 0:
    MATH_CONFIGS = [
        'algebra', 'counting_and_probability', 'geometry',
        'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus'
    ]

    added_math = 0
    seen_math_problems = set()
    for config in MATH_CONFIGS:
        if added_math >= remaining_for_math:
            break

        math_ds = load_dataset("EleutherAI/hendrycks_math", config, trust_remote_code=True)

        for split in MATH_SPLITS:
            if split not in math_ds:
                continue

            rows = math_ds[split]
            for rec in tqdm(rows, desc=f"Normalizing MATH/{config}/{split}"):
                if added_math >= remaining_for_math:
                    break
                item = normalize_math_record(rec, global_index)
                if item and item.get("problem") and item.get("solution_steps"):
                    prob_key = item["problem"]
                    if prob_key not in seen_math_problems:
                        seen_math_problems.add(prob_key)
                        all_records.append(item)
                        global_index += 1
                        added_math += 1

    source_counts["math"] = len([r for r in all_records if r["source"] == "math"])
    print("Loaded MATH examples:", source_counts["math"])
else:
    source_counts["math"] = 0
    print(f"Skipping MATH. (limit: {remaining_for_math})")


Normalizing MATH/algebra/train:   0%|          | 0/1744 [00:00<?, ?it/s]

Normalizing MATH/algebra/test:   0%|          | 0/1187 [00:00<?, ?it/s]

Loaded MATH examples: 1500


## Basic cleanup and export

In [ ]:
# We want to remove duplicates across sources as well, so we enforce uniqueness on the problem statements again before the final selection step.
seen = set()
deduped = []
for rec in all_records:
    key = (rec.get("source", ""), rec.get("problem", ""))
    if key in seen:
        continue
    seen.add(key)
    deduped.append(rec)

all_records = deduped

# Enforce the requested final source mix before writing.
selected_records = []
for source, target_count in TARGET_SOURCE_COUNTS.items():
    source_records = [r for r in all_records if r.get("source") == source]
    if len(source_records) < target_count:
        raise ValueError(
            f"Need {target_count} examples from {source}, but only found {len(source_records)} after normalization/dedup."
        )
    selected_records.extend(source_records[:target_count])

all_records = selected_records

for i, rec in enumerate(all_records):
    rec["example_index"] = i

# We can export the final normalized dataset to JSONL, where each line is a JSON object with the fields:
# - problem: the natural language problem statement
# - solution_steps: a list of strings representing the step-by-step solution
# - final_answer: the final answer extracted from the solution (if available)

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in all_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

assert len(all_records) == TARGET_TOTAL_PROBLEMS
print(f"Wrote {len(all_records):,} examples to {OUTPUT_JSONL}")
print("Counts by source:")
print(pd.Series([r["source"] for r in all_records]).value_counts())


Wrote 2,000 examples to normalized_outputs/solver_full_trajectory_dataset.jsonl
Counts by source:
math                 1500
naturalproofs         225
proofnet              225
demi_mathanalysis      50
Name: count, dtype: int64


In [ ]:
# Additionally, we can also export a "proof-only" version of the dataset that only includes the sources with step-by-step proofs (excluding MATH which often has more concise solutions), for use in settings where we want to focus on proof generation quality without the final answer extraction aspect.


proof_only = [r for r in all_records if r["source"] in {"demi_mathanalysis", "proofnet", "naturalproofs"}]
with open(OUTPUT_DIR / "solver_full_trajectory_proof_only.jsonl", "w", encoding="utf-8") as f:
    for rec in proof_only:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(len(proof_only))

500


## Inspect a few normalized examples

In [50]:

for rec in all_records[:5]:
    print("=" * 100)
    print("source:", rec["source"])
    print("problem:", rec["problem"][:400])
    print("num_steps:", len(rec["solution_steps"]))
    print("first_step:", rec["solution_steps"][0][:300] if rec["solution_steps"] else "")
    print("final_answer:", rec.get("final_answer", ""))


source: math
problem: Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).
num_steps: 6
first_step: For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$.
final_answer: 0
source: math
problem: A rectangular band formation is a formation with $m$ band members in each of $r$ rows, where $m$ and $r$ are integers. A particular band has less than 100 band members. The director arranges them in a rectangular formation and finds that he has two members left over. If he increases the number of members in each row by 1 and reduces the number of rows by 2, there are exactly enough places in the n
num_steps: 8
first_step: Let $x$ be the number of band members in each row for the original formation, when two are left over.
final_answer: 98
so